# Boston Housing Prediction Analysis

**Tabular Regression Project** — Predict Boston housing prices with focus on prediction accuracy.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Boston Housing (506 rows × 14 features)
Target: `medv`

## 2 · Project Overview

This project focuses on **maximizing prediction accuracy** for Boston housing prices. We use the local `Boston Dataset.csv` (506 rows × 14 columns) and apply systematic feature engineering and model tuning to achieve the best RMSE and R² possible.

## 3 · Learning Objectives

1. Build an optimized regression pipeline on the Boston dataset.
2. Apply feature engineering to improve prediction.
3. Compare boosting models with different hyperparameters.
4. Use LazyPredict and FLAML for systematic model search.
5. Achieve strong R² through iterative improvement.

## 4 · Problem Statement

Predict **median house value** (`medv`) with the best possible accuracy using 13 features describing Boston suburbs.

## 5 · Why This Project Matters

- Teaches the full ML workflow: EDA → preprocessing → modeling → evaluation.
- Builds intuition for when feature engineering helps vs. hurts.
- Demonstrates diminishing returns on a small dataset.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 506 |
| **Columns** | 15 (13 features + target + index) |
| **Target** | medv |
| **File** | `Boston Dataset.csv` (local) |

## 7 · Dataset Source and License Notes

- **Source**: UCI ML Repository.
- **License**: Public domain.
- **File**: Local CSV included in the project folder.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'medv'
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
DATA_PATH = os.path.join(SAVE_DIR, 'Boston Dataset.csv')
assert os.path.exists(DATA_PATH), f'Dataset not found at {DATA_PATH}'
df = pd.read_csv(DATA_PATH)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
print(f'Loaded: {df.shape}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Target: min={df[TARGET].min()}, max={df[TARGET].max()}, mean={df[TARGET].mean():.2f}')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].scatter(df['rm'], df[TARGET], alpha=0.5, s=10)
axes[0,0].set_xlabel('rm'); axes[0,0].set_ylabel(TARGET)
axes[0,0].set_title('Rooms vs Price')

axes[0,1].scatter(df['lstat'], df[TARGET], alpha=0.5, s=10)
axes[0,1].set_xlabel('lstat'); axes[0,1].set_ylabel(TARGET)
axes[0,1].set_title('Lower Status % vs Price')

df[TARGET].hist(bins=30, ax=axes[1,0], edgecolor='black')
axes[1,0].set_title('Price Distribution')

corr_target = df.corr()[TARGET].drop(TARGET).sort_values()
corr_target.plot(kind='barh', ax=axes[1,1])
axes[1,1].set_title('Correlation with medv')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkew: {df[TARGET].skew():.2f}')
print(f'Capped at 50: {(df[TARGET] == 50).sum()} rows')

## 15 · Train / Validation / Test Split

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 16 · Preprocessing Strategy

All numeric features. No missing values. We engineer additional features.

## 17 · Feature Engineering

In [ ]:
for df_part in [X_train, X_test]:
    df_part['rm_sq'] = df_part['rm'] ** 2
    df_part['lstat_log'] = np.log1p(df_part['lstat'])
    df_part['rm_lstat'] = df_part['rm'] * df_part['lstat']
    df_part['tax_per_room'] = df_part['tax'] / (df_part['rm'] + 0.1)

print(f'Features: {X_train.shape[1]}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['CatBoost']

y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- Feature engineering (rm², log(lstat), rm×lstat) helps capture non-linear relationships.
- Boosting models naturally handle these non-linearities.
- The capped medv=50 values are consistently under-predicted.
- For real estate pricing, R² > 0.85 is achievable with these features.

## 26 · Limitations

- Only 506 rows.
- Censored target at medv=50.
- Ethical concerns with the `black` variable.
- 1970s data — not reflective of modern Boston.

## 27 · How to Improve This Project

1. Exclude censored rows.
2. Hyperparameter tune with Optuna.
3. Stack top models.
4. Use cross-validation.

## 28 · Production Considerations

- Current data needed.
- Fair lending compliance.
- Regular retraining schedule.
- Prediction intervals for valuation uncertainty.

## 29 · Common Mistakes

1. Not handling censored target.
2. Over-engineering features on 506 rows.
3. Not validating with cross-validation.
4. Ignoring ethical concerns.

## 30 · Mini Challenge / Exercises

1. Try log-transforming the target.
2. Remove rows with medv=50 and compare.
3. Add polynomial features of degree 2 for top 3 features.
4. Use 5-fold CV instead of single split.

## 31 · Final Summary / Key Takeaways

- Feature engineering improves linear models; boosting models handle non-linearity naturally.
- rm and lstat remain the key features.
- This small dataset is a great learning tool but limited for production use.
- Systematic model comparison (LazyPredict + FLAML + manual) is best practice.

In [ ]:
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['Baseline_LR'] = all_results['Baseline_LR']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')